In [1]:
import pandas as pd
fetched_commits = pd.read_json('../datasets/raw/eligible_ai_pop_commits_metadata.jsonl', lines=True)
pull_requests = pd.read_parquet('../datasets/aidev/pull_request.parquet')
    
pull_request_ids = (
    pull_requests[['id', 'html_url']]
    .dropna(subset=['id', 'html_url'])
    .drop_duplicates(subset=['html_url'])
    .rename(columns={'id': 'pr_id', 'html_url': 'pr_url'})
)

fetched_commits['pr_url'] = fetched_commits['pr_url'].str.rstrip('/')
pull_request_ids['pr_url'] = pull_request_ids['pr_url'].str.rstrip('/')

fetched_commits = fetched_commits.drop(columns=['pr_id'], errors='ignore').merge(
    pull_request_ids,
    on='pr_url',
    how='left',
)
print(f'Number of fetched commits {len(fetched_commits)}')
fetched_commits.head(5)

Number of fetched commits 2833


,sha,pr_url,commit_details,pr_id
0,9325e444ec074e4c1b636b52c6492217eec7bf23,https://github.com/mlflow/mlflow/pull/16658,{'sha': '9325e444ec074e4c1b636b52c6492217eec7b...,3214555104
1,4cbfa276acb89dec117d822b956909e0d54dc51d,https://github.com/umputun/spot/pull/258,{'sha': '4cbfa276acb89dec117d822b956909e0d54dc...,2883609845
2,5943104753be2820df8bac1fe123644bd3f461e6,https://github.com/nrwl/nx/pull/31380,{'sha': '5943104753be2820df8bac1fe123644bd3f46...,3098972655
3,7916be281fa7a9f8e0dfe31e75f8759143881a25,https://github.com/nrwl/nx/pull/31380,{'sha': '7916be281fa7a9f8e0dfe31e75f8759143881...,3098972655
4,0a8bf204f723f81c8df376993827373e19d74e3d,https://github.com/JessicaTegner/pypandoc/pull...,{'sha': '0a8bf204f723f81c8df376993827373e19d74...,3146518192


In [2]:
fetched_commits_normalized = pd.json_normalize(fetched_commits['commit_details'])
fetched_eligible_commits = pd.concat([fetched_commits[['pr_id','pr_url']],fetched_commits_normalized],axis=1)
fetched_eligible_commits

,pr_id,pr_url,sha,node_id,url,html_url,comments_url,parents,files,commit.author.name,...,committer.events_url,committer.received_events_url,committer.type,committer.user_view_type,committer.site_admin,stats.total,stats.additions,stats.deletions,author,committer
0,3214555104,https://github.com/mlflow/mlflow/pull/16658,9325e444ec074e4c1b636b52c6492217eec7bf23,C_kwDOCB5Jx9oAKDkzMjVlNDQ0ZWMwNzRlNGMxYjYzNmI1...,https://api.github.com/repos/mlflow/mlflow/com...,https://github.com/mlflow/mlflow/commit/9325e4...,https://api.github.com/repos/mlflow/mlflow/com...,[{'sha': '44a7e27458d789351de33db399d0925c3303...,[{'sha': '45935a3e123e096d91a981585b1f79dd4868...,harupy,...,https://api.github.com/users/harupy/events{/pr...,https://api.github.com/users/harupy/received_e...,User,public,False,620,620,0,NaN,NaN
1,2883609845,https://github.com/umputun/spot/pull/258,4cbfa276acb89dec117d822b956909e0d54dc51d,C_kwDOJZfWAtoAKDRjYmZhMjc2YWNiODlkZWMxMTdkODIy...,https://api.github.com/repos/umputun/spot/comm...,https://github.com/umputun/spot/commit/4cbfa27...,https://api.github.com/repos/umputun/spot/comm...,[{'sha': '44f1ceb3cd393612c615d137e965cad041bd...,[{'sha': 'de96fbf31438a2f5ef60d2f5e9f8f0426d51...,Umputun,...,https://api.github.com/users/umputun/events{/p...,https://api.github.com/users/umputun/received_...,User,public,False,643423,331256,312167,NaN,NaN
2,3098972655,https://github.com/nrwl/nx/pull/31380,5943104753be2820df8bac1fe123644bd3f461e6,C_kwDOBfbSFNoAKDU5NDMxMDQ3NTNiZTI4MjBkZjhiYWMx...,https://api.github.com/repos/nrwl/nx/commits/5...,https://github.com/nrwl/nx/commit/5943104753be...,https://api.github.com/repos/nrwl/nx/commits/5...,[{'sha': '7595d8a5e11c1c8ca339922026efeca7bc43...,[{'sha': 'bcd8ef5933f3e1779e11690da81beb5844ad...,Jason Jean,...,https://api.github.com/users/FrozenPandaz/even...,https://api.github.com/users/FrozenPandaz/rece...,User,public,False,37,37,0,NaN,NaN
3,3098972655,https://github.com/nrwl/nx/pull/31380,7916be281fa7a9f8e0dfe31e75f8759143881a25,C_kwDOBfbSFNoAKDc5MTZiZTI4MWZhN2E5ZjhlMGRmZTMx...,https://api.github.com/repos/nrwl/nx/commits/7...,https://github.com/nrwl/nx/commit/7916be281fa7...,https://api.github.com/repos/nrwl/nx/commits/7...,[{'sha': '5943104753be2820df8bac1fe123644bd3f4...,[{'sha': 'd7a814271e57391d7758156169e64d9c8c57...,FrozenPandaz,...,https://api.github.com/users/FrozenPandaz/even...,https://api.github.com/users/FrozenPandaz/rece...,User,public,False,658,638,20,NaN,NaN
4,3146518192,https://github.com/JessicaTegner/pypandoc/pull...,0a8bf204f723f81c8df376993827373e19d74e3d,C_kwDOACqFp9oAKDBhOGJmMjA0ZjcyM2Y4MWM4ZGYzNzY5...,https://api.github.com/repos/JessicaTegner/pyp...,https://github.com/JessicaTegner/pypandoc/comm...,https://api.github.com/repos/JessicaTegner/pyp...,[{'sha': '162c8b7df2f9e06094204c82d02de5556fb6...,[{'sha': '92534ccd9c8056d41c9da3790730d141f7ae...,JessicaTegner,...,https://api.github.com/users/JessicaTegner/eve...,https://api.github.com/users/JessicaTegner/rec...,User,public,False,6,3,3,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2828,3271196926,https://github.com/AgentOps-AI/agentops/pull/1179,c961ce0047cbde04ad9eb43126db56850c216518,C_kwDOKHj8MdoAKGM5NjFjZTAwNDdjYmRlMDRhZDllYjQz...,https://api.github.com/repos/AgentOps-AI/agent...,https://github.com/AgentOps-AI/agentops/commit...,https://api.github.com/repos/AgentOps-AI/agent...,[{'sha': '62109b509c33b735d38564d73dca9ffd797e...,[{'sha': 'b946987d9125062c1d983d2ae04151477b10...,Devin AI,...,https://api.github.com/users/devin-ai-integrat...,https://api.github.com/users/devin-ai-integrat...,Bot,public,False,28,19,9,NaN,NaN
2829,3271196926,https://github.com/AgentOps-AI/agentops/pull/1179,a69ec0295fe9057cc61ac997575e5cf11fd0b364,C_kwDOKHj8MdoAKGE2OWVjMDI5NWZlOTA1N2NjNjFhYzk5...,https://api.github.com/repos/AgentOps-AI/agent...,https://github.com/AgentOps-AI/agentops/commit...,https://api.github.com/repos/AgentOps-AI/agent...,[{'sha': 'c961ce0047cbde04ad9eb43126db56850c21...,

In [3]:
fetched_eligible_commits.to_csv('../datasets/processed/fetched_eligible_pop_commits.csv')

# PRs

In [5]:
# get prs of the fetched commits
fetched_prs = fetched_eligible_commits[['pr_url']].dropna().drop_duplicates()
fetched_prs
print(len(fetched_prs))

# extract repository from pr_url
fetched_prs['repository'] = fetched_prs['pr_url'].apply(lambda x: x.split('/pull/')[0] if pd.notnull(x) else x)
print(len(fetched_prs.groupby('repository')))

1032
281
